# Data normalization
Use the z-score normalization. Also include ethnicity and gender processing.


In [ ]:
import pandas as pd
import os
from pathlib import Path
import numpy as np

In [ ]:
# load data
time_resolution = "2h"
interested_split = 'test'
impute_method = "bffill" # choose from bffill, global, linearinterp

if impute_method == "bffill":
    imputed_dir = f"data/MIMICIII_last48h_ts{time_resolution}/imputed"
    export_dir = f"data/MIMICIII_last48h_ts{time_resolution}"  # This is the final step so we will export the data to the root directory
else:
    imputed_dir = f"data/MIMICIII_last48h_ts{time_resolution}_impute-{impute_method}/imputed"
    export_dir = f"data/MIMICIII_last48h_ts{time_resolution}_impute-{impute_method}"  # This is the final step so we will export the data to the root directory

Path(os.path.join(export_dir, interested_split)).mkdir(exist_ok=True, parents=True)

demo_path = os.path.join(imputed_dir, interested_split,'demographics.csv')
ts_path = os.path.join(imputed_dir, interested_split,'time-series.csv')
label_path = os.path.join(imputed_dir, interested_split,'label.csv')

train_demo_path = os.path.join(imputed_dir, 'train','demographics.csv')
train_ts_path = os.path.join(imputed_dir, 'train','time-series.csv')

demo_export_path = os.path.join(export_dir, interested_split,'demographics.csv')
ts_export_path = os.path.join(export_dir, interested_split,'time-series.csv')
label_export_path = os.path.join(export_dir, interested_split,'label.csv')

demo_df = pd.read_csv(demo_path)
ts_df = pd.read_csv(ts_path)
label_df = pd.read_csv(label_path)

train_demo_df = pd.read_csv(train_demo_path)
train_ts_df = pd.read_csv(train_ts_path)

print(f"hadm_id num of demo_df:{demo_df['hadm_id'].unique().shape[0]}")
print(f"hadm_id num of vital_df:{ts_df['hadm_id'].unique().shape[0]}")
print(f"hadm_id num of label_df:{label_df['hadm_id'].unique().shape[0]}")

## Normalize

In [ ]:
# get the min, max, mean of each feature
demo_features = ['age', "staytime"]

train_demo_mean = train_demo_df.loc[:, demo_features].mean().values
train_demo_std = train_demo_df.loc[:, demo_features].std().values
train_demo_cv = train_demo_std / train_demo_mean

train_ts_mean = train_ts_df.iloc[:, 3:].mean().values.reshape(1, -1)
train_ts_std = train_ts_df.iloc[:, 3:].std().values.reshape(1, -1)
train_ts_cv = train_ts_std / train_ts_mean

print(f"train_demo_cv:{train_demo_mean}")
print(f"train_ts_cv:{train_ts_mean}")

In [ ]:
# apply the z-score normalization
demo_df.loc[:, demo_features] = ((demo_df.loc[:, demo_features] - train_demo_mean) / train_demo_std)
ts_df.iloc[:, 3:] = (ts_df.iloc[:, 3:] - train_ts_mean) / train_ts_std

demo_df.head()

In [ ]:
ts_df.head()

In [ ]:
# export the processed data
demo_df.to_csv(demo_export_path, index=False)
ts_df.to_csv(ts_export_path, index=False)
label_df.to_csv(label_export_path, index=False)